# Child Anemia in Peru: reproducible ENDES workflow

This notebook runs the same real-data workflow documented in the repository. It is a reproducibility exercise, not a clinical prediction tool. Run the cells in order and do not skip the data retrieval step.

## Before starting

You need access to the project's shared Google Drive DVC folder. Credentials stay in your own Colab session and must never be added to GitHub. If Google blocks DVC's shared OAuth application, use the private service-account route documented in `05_pipeline/README.md` before running the retrieval cell.

In [1]:
from pathlib import Path
import os
import subprocess

REPO_NAME = 'deivhy-torres-vargas-unmsm'
COLAB_REPO = Path('/content') / REPO_NAME
if Path('/content').exists():
    if not COLAB_REPO.exists():
        subprocess.run(['git', 'clone', f'https://github.com/kedec123/{REPO_NAME}.git', str(COLAB_REPO)], check=True)
    REPO_DIR = COLAB_REPO
else:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / REPO_NAME]
    REPO_DIR = next((item for item in candidates if (item / '05_pipeline').exists()), None)
    if REPO_DIR is None:
        raise FileNotFoundError('Run this notebook from the repository or use Google Colab.')
os.chdir(REPO_DIR)
print(f'Working directory: {Path.cwd()}')
subprocess.run(['git', 'status', '--short'], check=True)

Working directory: repository root


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

In [2]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '05_pipeline/requirements.txt'], check=True)
print('Dependencies installed. If Colab asks for a restart, restart the session and rerun from the first cell.')

Dependencies installed. If Colab asks for a restart, restart the session and rerun from the first cell.


In [3]:
from pathlib import Path
import os
import subprocess

if 'REPO_DIR' not in globals() or not REPO_DIR.exists():
    raise FileNotFoundError('Repository folder is missing. Rerun the first setup cell.')
os.chdir(REPO_DIR)
data_path = REPO_DIR / '05_pipeline' / 'data' / 'endes_anemia_children_2019_2024.csv'
if data_path.exists():
    print('Analytical CSV is already available; DVC retrieval skipped.')
else:
    subprocess.run([sys.executable, '-m', 'dvc', 'pull'], cwd=REPO_DIR / '05_pipeline', check=True)
    print('DVC retrieval finished.')

Analytical CSV is already available; DVC retrieval skipped.


## Validate the analytical CSV

The file contains de-identified records for children aged 6-35 months. It uses the legacy ENDES outcome for a consistent 2019-2024 trend; the updated 2024 field is retained for sensitivity discussion only.

In [4]:
from pathlib import Path
import pandas as pd

data_path = Path('05_pipeline/data/endes_anemia_children_2019_2024.csv')
if not data_path.exists():
    raise FileNotFoundError('The CSV is missing. Rerun dvc pull and check Drive access.')

df = pd.read_csv(data_path, dtype={'department_code': 'string'})
print(f'Rows: {len(df):,}')
print(f'Years: {sorted(df.survey_year.unique())}')
print(f'Age range: {df.age_months.min():.0f}-{df.age_months.max():.0f} months')
df.head()

Rows: 57,539
Years: [2019, 2020, 2021, 2022, 2023, 2024]
Age range: 6-35 months


,analysis_id,survey_year,age_months,hemoglobin_g_dl,hemoglobin_adjusted_legacy_g_dl,hemoglobin_adjusted_new_2024_g_dl,anemia_legacy_level,anemia_new_2024_level,anemia_legacy,child_sex_code,mother_education_code,wealth_quintile,residence_code,survey_weight,cluster_code,stratum_code,department_code
0,2019_00001,2019,8.0,132.0,121.0,NaN,4.0,NaN,0,1,3.0,4,1,0.154803,1,3,01
1,2019_00002,2019,18.0,137.0,126.0,NaN,4.0,NaN,0,2,3.0,4,1,0.154803,1,3,01
2,2019_00003,2019,8.0,98.0,87.0,NaN,2.0,NaN,1,2,3.0,5,1,0.154803,1,3,01
3,2019_00004,2019,15.0,117.0,106.0,NaN,3.0,NaN,1,2,2.0,5,1,0.154803,1,3,01
4,2019_00005,2019,13.0,125.0,114.0,NaN,4.0,NaN,0,2,3.0,4,1,0.154803,1,3,01


In [5]:
subprocess.run([sys.executable, '05_pipeline/src/analyze_endes.py'], check=True)
subprocess.run([sys.executable, '05_pipeline/src/survey_analysis.py'], check=True)
subprocess.run([sys.executable, '05_pipeline/src/sensitivity_2024.py'], check=True)
display(pd.read_csv('05_pipeline/docs/analysis_by_year_with_ci.csv'))
display(pd.read_csv('05_pipeline/docs/sensitivity_2024.csv'))

,survey_year,sample_size,weighted_anemia_prevalence,ci_95_lower,ci_95_upper,bootstrap_replicates
0,2019,10320,0.398324,0.384903,0.409429,300
1,2020,6051,0.382516,0.365445,0.400596,300
2,2021,10902,0.384404,0.373898,0.395584,300
3,2022,10557,0.419939,0.409413,0.431039,300
4,2023,9951,0.427827,0.414696,0.440647,300
5,2024,9758,0.435072,0.423180,0.445498,300


,definition,weighted_anemia_prevalence
0,Legacy comparable HW57,0.435072
1,Updated 2024 HW57A,0.349234


### Models and preprocessing

The main study is a weighted repeated cross-sectional analysis. The separate machine-learning exercise is a tabular binary-classification task: `anemia_legacy` is 1 for anemia and 0 for no anemia. Logistic Regression is an interpretable baseline; Random Forest and Extra Trees are nonlinear ensemble comparisons. Numeric features use median imputation and standardization; categorical features use most-frequent imputation and one-hot encoding.

In [6]:
import sys

sys.path.insert(0, '05_pipeline/src')
from train import CATEGORICAL_FEATURES, NUMERIC_FEATURES, build_pipeline

print('Problem: binary classification (anemia / no anemia)')
print(f'Numeric features: {NUMERIC_FEATURES}')
print(f'Categorical features: {CATEGORICAL_FEATURES}')
for model_name in ('logistic_regression', 'random_forest', 'extra_trees'):
    model = build_pipeline(model_name, seed=42).named_steps['model']
    print(f'{model_name}: {model.__class__.__name__}')

Problem: binary classification (anemia / no anemia)
Numeric features: ['age_months', 'mother_education_code', 'wealth_quintile']
Categorical features: ['child_sex_code', 'residence_code', 'department_code', 'survey_year']
logistic_regression: LogisticRegression
random_forest: RandomForestClassifier
extra_trees: ExtraTreesClassifier


## Run exploratory experiments

The next cell evaluates three models on five prespecified stratified 80/20 splits and writes MLflow records. It reports the full split-level results and their mean, standard deviation, minimum, and maximum; it does not estimate clinical usefulness.

In [7]:
subprocess.run([sys.executable, '05_pipeline/src/run_experiments.py'], check=True)
results = pd.read_csv('05_pipeline/docs/experiment_results.csv')
summary = pd.read_csv('05_pipeline/docs/experiment_summary.csv')
display(summary.round(4))
results

,model,auc_roc_mean,auc_roc_std,auc_roc_min,auc_roc_max,pr_auc_mean,pr_auc_std,pr_auc_min,pr_auc_max,accuracy_mean,...,accuracy_min,accuracy_max,f1_mean,f1_std,f1_min,f1_max,recall_mean,recall_std,recall_min,recall_max
0,extra_trees,0.7050,0.0027,0.7008,0.7078,0.6402,0.0022,0.6366,0.6421,0.6538,...,0.6514,0.6557,0.6223,0.0022,0.6193,0.6253,0.6480,0.0050,0.6422,0.6534
1,logistic_regression,0.7077,0.0038,0.7021,0.7117,0.6377,0.0021,0.6349,0.6408,0.6563,...,0.6509,0.6614,0.6249,0.0043,0.6205,0.6300,0.6506,0.0048,0.6442,0.6556
2,random_forest,0.7051,0.0035,0.7001,0.7085,0.6401,0.0033,0.6364,0.6452,0.6540,...,0.6507,0.6560,0.6193,0.0007,0.6187,0.6205,0.6396,0.0027,0.6376,0.6444


,seed,model,sample_size,test_size,auc_roc,pr_auc,accuracy,f1,recall
0,13,logistic_regression,57539,11508,0.711708,0.637779,0.661366,0.630020,0.655213
1,13,random_forest,57539,11508,0.708472,0.640647,0.655978,0.620458,0.639021
2,13,extra_trees,57539,11508,0.707812,0.639581,0.655718,0.622882,0.646130
3,21,logistic_regression,57539,11508,0.707872,0.640825,0.654067,0.621038,0.644155
4,21,random_forest,57539,11508,0.707719,0.645237,0.655023,0.619586,0.638428
5,21,extra_trees,57539,11508,0.704781,0.642095,0.652503,0.619252,0.642180
6,42,logistic_regression,57539,11508,0.706305,0.637452,0.656239,0.624383,0.649289
7,42,random_forest,57539,11508,0.702853,0.638118,0.654675,0.619057,0.637638
8,42,extra_trees,57539,11508,0.704607,0.640519,0.653893,0.621280,0.645142
9,87,logistic_regression,57539,11508,0.702082,0.634924,0.650852,0.620514,0.648697


In [8]:
subprocess.run([sys.executable, '11_bias_audit/bias_audit.py'], check=True)
print(Path('11_bias_audit/bias_audit_report.md').read_text()[:1200])

# Fairness Audit: Adult Census Benchmark

This audit uses Fairlearn's Adult Census dataset, not ENDES. It is included to meet the course requirement for a reproducible fairness exercise on a standard benchmark. The sensitive feature is recorded sex. The model deliberately excludes sex from training, then evaluates group differences by sex. The favourable label for this exercise is income above 50K; this convention is specific to the benchmark and does not describe a health outcome.

## Labels before modelling

The first table checks whether the benchmark label already differs by the sensitive feature. Disparate impact is the smaller group selection rate divided by the larger one. A value closer to 1 indicates similar selection rates; it does not establish that the data-generating process is fair.

|   demographic_parity_difference |   disparate_impact |
|--------------------------------:|-------------------:|
|                          0.1945 |             0.3597 |

| sex    |   sample

## Optional: inspect MLflow

In a local environment, run `mlflow ui --backend-store-uri ./mlruns` and open `http://127.0.0.1:5000`. Colab does not expose that local address directly without a tunnelling method, so the saved CSV and `mlruns/` artefact are the portable evidence in this notebook.